In [2]:
import pandas as pd
from pathlib import Path

print("Notebook kernel is working.")

Notebook kernel is working.


In [3]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "SP500_Historical_Data.csv"

print("Dataset path:", DATA_PATH)
print("File found:", DATA_PATH.exists())

Dataset path: e:\Quantum Project\data\raw\SP500_Historical_Data.csv
File found: True


In [5]:
df = pd.read_csv(DATA_PATH)
df.sample(5, random_state = 42)

,Ticker,Date,Open,High,Low,Close,Adj Close,Volume
966771,EXPE,2023-12-27,152.11,152.81,150.49,152.02,152.02,1420100
117642,ALB,2016-07-01,69.86,72.35,69.86,71.51,71.51,1588400
2393821,TPR,2021-04-16,41.14,41.42,40.67,41.13,41.13,2874100
859613,EME,2022-04-28,111.86,113.17,103.86,104.15,104.15,567100
877865,EOG,2002-08-15,5.94,6.11,5.94,6.08,6.08,3214800


In [6]:
print("Rows and columns:", df.shape)

print("\nColumn types:")
print(df.dtypes)

Rows and columns: (2703531, 8)

Column types:
Ticker           str
Date             str
Open         float64
High         float64
Low          float64
Close        float64
Adj Close    float64
Volume         int64
dtype: object


In [8]:
df["Date"] = pd.to_datetime(df["Date"], errors="raise")
print("Unique stocks:", df["Ticker"].nunique())
print("Earliest date:", df["Date"].min().date())
print("Latest date:", df["Date"].max().date())

Unique stocks: 472
Earliest date: 2000-01-03
Latest date: 2026-02-20


In [9]:
print("Missing values per column:")
print(df.isna().sum())

duplicate_rows = df.duplicated(subset=["Ticker", "Date"]).sum()
print("\nDuplicate Ticker-Date rows:", duplicate_rows)

Missing values per column:
Ticker       0
Date         0
Open         0
High         0
Low          0
Close        0
Adj Close    0
Volume       0
dtype: int64

Duplicate Ticker-Date rows: 0


In [10]:
price_columns = ["Open", "High", "Low", "Close", "Adj Close"]

invalid_prices = (df[price_columns] <= 0).any(axis=1).sum()
negative_volume = (df["Volume"] < 0).sum()

invalid_high = (
    df["High"] < df[["Open", "Close", "Low"]].max(axis=1)
).sum()

invalid_low = (
    df["Low"] > df[["Open", "Close", "High"]].min(axis=1)
).sum()

print("Rows with invalid prices:", invalid_prices)
print("Rows with negative volume:", negative_volume)
print("Rows with invalid High values:", invalid_high)
print("Rows with invalid Low values:", invalid_low)

Rows with invalid prices: 0
Rows with negative volume: 0
Rows with invalid High values: 0
Rows with invalid Low values: 1


In [11]:
invalid_low_rows = df[
    df["Low"] > df[["Open", "Close", "High"]].min(axis=1)
]

invalid_low_rows

,Ticker,Date,Open,High,Low,Close,Adj Close,Volume
1269082,HUBB,2021-05-05,181.34,183.8,181.55,183.33,183.33,127234


In [12]:
corrections = invalid_low_rows.copy()

corrections["Corrected Low"] = corrections[
    ["Open", "High", "Low", "Close"]
].min(axis=1)

corrections[
    ["Ticker", "Date", "Open", "High", "Low", "Close", "Corrected Low"]
]

,Ticker,Date,Open,High,Low,Close,Corrected Low
1269082,HUBB,2021-05-05,181.34,183.8,181.55,183.33,181.34


In [13]:
df.loc[invalid_low_rows.index, "Low"] = corrections["Corrected Low"]

In [14]:
remaining_invalid_low = (
    df["Low"] > df[["Open", "Close", "High"]].min(axis=1)
).sum()

print("Remaining invalid Low values:", remaining_invalid_low)

Remaining invalid Low values: 0


In [15]:
df = df.sort_values(["Ticker", "Date"]).reset_index(drop=True)

df[["Ticker", "Date", "Close"]].head()

,Ticker,Date,Close
0,A,2000-01-03,43.04
1,A,2000-01-04,39.75
2,A,2000-01-05,37.28
3,A,2000-01-06,35.86
4,A,2000-01-07,38.85


In [16]:
ticker_summary = (
    df.groupby("Ticker")["Date"]
    .agg(Start_Date="min", End_Date="max", Trading_Days="count")
    .sort_values("Trading_Days", ascending=False)
)

ticker_summary.head(10)

,Start_Date,End_Date,Trading_Days
Ticker,,,
ABT,2000-01-03,2026-02-20,6573
ZBRA,2000-01-03,2026-02-20,6573
A,2000-01-03,2026-02-20,6573
ACGL,2000-01-03,2026-02-20,6573
AAPL,2000-01-03,2026-02-20,6573
ADBE,2000-01-03,2026-02-20,6573
WDC,2000-01-03,2026-02-20,6573
WAT,2000-01-03,2026-02-20,6573
WHR,2000-01-03,2026-02-20,6573


In [17]:
ticker_summary["Trading_Days"].describe()

count     472.000000
mean     5727.819915
std      1570.555355
min       256.000000
25%      5432.000000
50%      6573.000000
75%      6573.000000
max      6573.000000
Name: Trading_Days, dtype: float64

In [18]:
MIN_TRADING_DAYS = 5000

eligible_tickers = ticker_summary[
    ticker_summary["Trading_Days"] >= MIN_TRADING_DAYS
].index

print("Eligible tickers:", len(eligible_tickers))
print("Excluded tickers:", len(ticker_summary) - len(eligible_tickers))

Eligible tickers: 372
Excluded tickers: 100


In [19]:
model_df = df[df["Ticker"].isin(eligible_tickers)].copy()

print("Rows in modelling dataset:", len(model_df))
print("Stocks in modelling dataset:", model_df["Ticker"].nunique())

Rows in modelling dataset: 2405879
Stocks in modelling dataset: 372


In [20]:
model_df = df[df["Ticker"].isin(eligible_tickers)].copy()

print("Rows in modelling dataset:", len(model_df))
print("Stocks in modelling dataset:", model_df["Ticker"].nunique())

Rows in modelling dataset: 2405879
Stocks in modelling dataset: 372


In [21]:
print("Earliest modelling date:", model_df["Date"].min().date())
print("Latest modelling date:", model_df["Date"].max().date())

model_df[["Ticker", "Date", "Close"]].head()

Earliest modelling date: 2000-01-03
Latest modelling date: 2026-02-20


,Ticker,Date,Close
0,A,2000-01-03,43.04
1,A,2000-01-04,39.75
2,A,2000-01-05,37.28
3,A,2000-01-06,35.86
4,A,2000-01-07,38.85


In [22]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED_PATH = PROCESSED_DIR / "SP500_cleaned_eligible_tickers.csv"

model_df.to_csv(PROCESSED_PATH, index=False)

print("Saved processed dataset to:")
print(PROCESSED_PATH)

Saved processed dataset to:
e:\Quantum Project\data\processed\SP500_cleaned_eligible_tickers.csv


In [23]:
file_size_mb = PROCESSED_PATH.stat().st_size / (1024 ** 2)

print("File exists:", PROCESSED_PATH.exists())
print(f"File size: {file_size_mb:.1f} MB")

File exists: True
File size: 123.1 MB


In [ ]:
print("Earliest modelling date:", model_df["Date"].min().date())
print("Latest modelling date:", model_df["Date"].max().date())

model_df[["Ticker", "Date", "Close"]].head()

Earliest modelling date: 2000-01-03
Latest modelling date: 2026-02-20


,Ticker,Date,Close
0,A,2000-01-03,43.04
1,A,2000-01-04,39.75
2,A,2000-01-05,37.28
3,A,2000-01-06,35.86
4,A,2000-01-07,38.85


In [ ]:
print("Earliest modelling date:", model_df["Date"].min().date())
print("Latest modelling date:", model_df["Date"].max().date())

model_df[["Ticker", "Date", "Close"]].head()

Earliest modelling date: 2000-01-03
Latest modelling date: 2026-02-20


,Ticker,Date,Close
0,A,2000-01-03,43.04
1,A,2000-01-04,39.75
2,A,2000-01-05,37.28
3,A,2000-01-06,35.86
4,A,2000-01-07,38.85


# Dataset Understanding and Preparation

## Dataset selected
This project uses S&P 500 constituent daily market data containing Open, High, Low, Close, Adjusted Close, and Volume information.

## Initial quality checks
- The raw dataset contains 2,703,531 rows across 472 tickers.
- It covers 2000-01-03 to 2026-02-20.
- No missing values or duplicate ticker-date records were found.
- One invalid Low value was identified for HUBB on 2021-05-05 and corrected only in the working dataset.
- The original raw CSV remains unchanged.

## Modelling dataset
Stocks with fewer than 5,000 trading days were excluded to ensure sufficient historical information for time-based training and testing.

The resulting modelling dataset contains:
- 372 eligible stocks
- 2,405,879 rows
- Date range: 2000-01-03 to 2026-02-20

## Next step
The next notebook will create leakage-safe financial features and the next-day UP/DOWN prediction target.